## Kalman Filter

This section reviews the design and implementation of the Kalman filter. Its purpose is to estimate the robot’s position at each step by combining information from odometry and the camera.

The Kalman filter first performs a prediction step, where it estimates the robot’s next position based on its current pose and motion commands. However, odometry is not perfectly accurate, and small errors quickly accumulate, causing the Thymio to drift from its true position.

The update (correction) step compensates for this drift. When a camera measurement is available, the filter incorporates it to correct the predicted pose, pulling the estimate back toward the robot’s real location. This continuous predict–correct cycle keeps the robot accurately localized as it moves toward its goal.

### 1. State Definition

The goal of the EKF is to estimate the 2D pose of the Thymio robot.  
We define the state vector as:

$$
\begin{bmatrix}
x \\
y \\
\theta
\end{bmatrix}
$$

With:

- $x$: robot position along the horizontal axis, in [mm] 
- $y$: robot position along the vertical axis, in [mm]
- $\theta$: robot orientation, wrapped to $[- \pi, \pi]$, in [rad]

And the inputs to the motion models being:

- $v$: the linear speed
- $w$: the angular speed

Since the Thymio does not directly measure these two quantities, they were computed in the **Motion Control** module and passed to the EKF as inputs.

### 2. Motion Prediction Model

The model is based on an Extended Kalman Filter (EKF) since our system is nonlinear.
The prediction step uses a unicycle/differential-drive kinematic model, which matches the physical structure and motion constraints of the Thymio robot.

The Thymio has exactly two actuated wheels, making it a differential-drive platform. Its motion; moving forward/backward, rotating on the spot, and following curved trajectories; is accurately described by the unicycle model. Other models, such as the bicycle model or full dynamic models, allow lateral motion or include forces the Thymio does not experience, and are therefore not appropriate here.

We follow the standard kinematic equations:

$$
\begin{aligned}
x_{k+1} &= x_k + T_s\, v \cos(\theta_k), \\[4pt]
y_{k+1} &= y_k + T_s\, v \sin(\theta_k), \\[4pt]
\theta_{k+1} &= \theta_k + T_s\, \omega.
\end{aligned}
$$


The presence of the nonlinear terms $\cos(\theta_k)$ and $\sin(\theta_k)$ makes the system nonlinear, which is why an EKF is required instead of a standard Kalman Filter.

Next, the Jacobian of the motion model with respect to the state is:

$$
A =
\begin{bmatrix}
1 & 0 & -T_s\, v \sin\theta \\
0 & 1 & \;\;T_s\, v \cos\theta \\
0 & 0 & 1
\end{bmatrix}
$$

Finally, the covariance prediction step is given by:

$$
P_{k+1|k} = A\, P_{k|k}\, A^\top + Q
$$

where $Q$ represents the process noise, modelling uncertainty in the motion prediction. \
The units are $[mm^2]$ for $\sigma_x^2$ and $\sigma_y^2$ and $[rad^2]$ for $\sigma_\theta^2$.

The complete implementation of this prediction step is provided in the following code section:

```python

def ekf_predict(x, P, v, omega, Ts, Q):
    
    ...
    
    # Nonlinear motion model (discrete-time)
    x_pos_pred = x_pos + Ts * v * np.cos(theta)          
    y_pos_pred = y_pos + Ts * v * np.sin(theta)          
    theta_pred = wrap_angle(theta + Ts * omega)          
                
    ... 

    # Jacobian of the motion model w.r.t. state (A matrix)
    A = np.array([
        [1, 0, -Ts * v * np.sin(theta)],                 
        [0, 1,  Ts * v * np.cos(theta)],                 
        [0, 0,  1]                                       
    ])

    # Covariance prediction using linearized model
    P_pred = A @ P @ A.T + Q                             

    return x_pred, P_pred      

```

This function takes as inputs the state vector $x = [x,\,y,\,\theta]^T$, the current covariance matrix $P$,
the linear and angular velocities $v$ and $\omega$, the sampling time $T_s$, and the process noise matrix $Q$.
It returns the predicted state $x_{\text{pred}}$ and the predicted covariance $P_{\text{pred}}$.


#### Covariance
The covariance matrix is a measure of the uncertainty and correlation between the state variables.
The diagonal terms represent the uncertainty of each state component, while the off-diagonal terms indicate
how errors in one variable influence the others. For example, during a turning motion, the uncertainty in
$\theta$ affects both $x$ and $y$, increasing their correlation.

If the covariance is large, the Kalman filter interprets the prediction as unreliable and therefore gives
more weight to the measurement. Conversely, if the covariance is small, the filter trusts the prediction more.

Source: [The Kalman Filter (18 of 55) What is a Covariance Matrix?](https://youtu.be/mYAsKbwqGv0?si=bKLiNsdeX1D0KAqB)

#### Angle Wrap
The prediction step also uses a helper function to wrap angles:

```python
def wrap_angle(a):
    """
    Wraps an angle to the range [-pi, pi].

    """
    return (a + np.pi) % (2 * np.pi) - np.pi
```
This ensures that all angle computations remain within the interval $[- \pi, \pi]$,
avoiding discontinuities that would otherwise destabilize the filter.






